# Myeloid: Sub-Annotation 

Clusters the myeloid compartment on the scVI latent space, assigns level-1 cell
states from cluster markers (logistic regression and TF-IDF ranking plus curated
marker panels), refines ambiguous clusters by subclustering, merges macrophage
states into another broad `cell_state_v2`, and summarises state composition per
sample and RECIST treatment stage.

In [ ]:
# !pip install --quiet scanpy leidenalg igraph scikit-learn
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/myeloid

In [ ]:
import os

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfTransformer

## Config

In [ ]:
input_path = "sub_Myeloid_raw_scvi_combine.h5ad"
output_path = "sub_Myeloid_raw_scvi_combine_annotated.h5ad" 

annotation_csv = "tables/myeloid_cell_state_annotation.csv"   # barcode -> labels, for re-use without re-clustering
figures_output_dir = "./figures"
tables_output_dir = "./tables"

counts_layer = "raw_counts"
sample_col = "ID_2"
recist_col = "recist_v3"

# --- clustering ---
global_rep = "X_scVI_n_latent_hvg_10"       # level-1 neighbours / UMAP / Leiden
subcluster_rep = "X_scVI_n_latent_10"       # subclustering (as in the original run; see notes)
global_n_neighbors = 25
subcluster_n_neighbors = 40
leiden_resolution = 0.8
seed = 0

liver_biopsy_ids = ["R008", "R011"]          # matched on obs["ID"]
sample_order = [
    "TN001_AN", "TN006_AN", "TN007_AN", "TN009_AN", "TN004_AN", "TN008_AN", "TN003_AN", "TN005_AN", "TN002_AN",
    "TN008", "TN004", "TN003", "TN002", "TN005", "TN007", "TN001", "TN006", "TN009",
    "O001", "E001", "P013", "P008", "P003", "P012", "P005", "P002", "P006", "P009", "P001", "P011",
    "D003", "D001", "D002",
    "R010", "R012", "R006", "R001", "R003", "R007", "R008", "R011", "R002", "R009", "R004", "R005",
]
count_axis_break = 250                       # y-axis break for the per-sample cell-count barplot

# --- marker panels guided by literature---
marker_genes = {
    "Alveolar": ["INHBA", "PPARG", "ITGAM", "ITGAX", "CSF2RA", "FABP4", "SERPING1", "CD52", "MME", "CES1", "GPD1", "CFD", "RBP4", "MARCO", "SIGLEC1", "CD33", "CD68"],
    "Interstitial": ["LGMN", "MARCKS"],
    "Interstitial_1": ["FOLR2", "SELENOP", "F13A1", "SLC40A1"],
    "Interstitial_2": ["SPP1", "PLA2G7", "MMP9", "HAMP", "LYVE1", "PLTP", "CD36"],
    "Interstitial_3": ["TLR2", "MERTK"],
    "Extra": ["STAB1"],
    "Recruited_mac": ["VCAN", "FN1", "CD14", "FCN1", "S100A8", "RNASE1", "EMP1", "FCGR3A", "TEK"],
    "cDC2": ["ENTPD1", "AFF3", "PAK1", "CD1E", "CD1C", "CLEC4A", "FCER1A", "CD207", "CLEC10A", "SIRPA"],
    "cDC1": ["CLEC9A", "FLT3", "CADM1", "IRF8", "XCR1"],
    "activated_DC": ["LAMP3", "CCL19", "CCR7", "FSCN1", "BIRC3", "CD274"],
    "pDC": ["IL3RA", "CLEC4C"],
    "myeloid dividing": ["TOP2A", "MKI67", "CENPP", "HMGB1"],
    "IFN_TAM": ["CXCL10", "ISG15", "IDO1", "STAT1", "STAT2", "CD86"],
    "Reg_TAM": ["ARG1", "MRC1", "CX3CR1", "CD274", "PDCD1", "HAVCR2", "TREM2", "PDCD1LG2"],
    "RTM_TAM": ["HES1", "CCL18", "CCL23", "FBP1", "LGALS3"],
    "LA_TAM": ["APOE", "FABP5", "APOC1", "ACP5", "MAF"],
    "Inflam_TAM": ["IL1B", "CCL3", "CXCL1", "CXCL2", "CXCL3", "CXCL5", "CXCL8", "NFKB1", "CCL3L1", "REL", "CCL5"],
    "Angio_TAM": ["VEGFA", "THBS1", "AREG"],
    "MHC": ["HLA-DRA", "HLA-DRB1", "HLA-DQA1"],
    "M2-like": ["IL4R", "STAT2", "STAT6", "CD163", "TGFB1", "CLEC7A", "IGF1", "CCL2"],
    "OxPhos": ["SLC25A20", "IDH2", "SUCLA2", "HADHB", "ACOX1", "H6PD", "PPT1", "HK3"],
    "APC": ["CD80", "CD86"],
    "CD16- classical mono": ["ITGAX", "FCGR1A", "CD52", "CD163"],
}

marker_genes_neutrophil = {
    "Macs": ["INHBA", "CD14", "CCR2", "CD68", "CD163", "MRC1", "FABP4", "MARCO", "PPARG", "SIGLEC1", "TGFB1", "IL10", "ARG1"],
    "Macs_rec": ["FCN1", "S100A9", "S100A8", "NOS2"],
    "Neutrophil tissue": ["FUT4", "CEACAM8", "ITGAM", "ICAM1", "ELANE", "MPO", "LTF", "MMP8"],
    "Neut immature suppressive": ["CXCR4", "MMP9", "ARG1", "OLR1", "CD24", "CEACAM1", "CD274", "IL1B"],
    "Neutrophil blood": ["CXCR2", "SELL", "MME"],
}


gene_panels = {
    "level1_markers": [          # level-1 clusters: macrophage / mast / MDSC / cDC / monocyte / neutrophil markers
        "leiden_scvi", "CD14", "INHBA", "FABP4", "TREM2", "PPARG", "MERTK", "MRC1", "TLR2",
        "STAB1", "SPP1", "APOE", "CCL18", "CD36",
        "KIT", "ENPP3", "F2RL1", "IL4R", "HLA-DRA", "HLA-DRB1",      # mast cell
        "IL1B", "CSF3R", "CXCR2", "ARG1",                          # MDSC
        "XCR1", "CLEC9A", "CADM1",                                 # cDC1
        "CD1C", "CD207", "CCR7", "FSCN1", "BIRC3",                 # cDC2 / mregDC
        "PLAC8", "VCAN", "FCN1", "S100A6", "C1QA", "CD68", "LYZ",
        "FUT4", "FCGR3A",                                          # CD15, CD16
        "ITGAM", "CEACAM8", "ELANE", "MPO", "CXCR2", "S100A9", "S100A9", "FCGR1A",  # neutrophil, CD64
    ],
    "level1_check": ["CD36", "IL4R", "TREM2", "ITGAM", "CD33", "FABP4", "MERTK", "STAB1", "SPP1", "FOLR2", "MMP9"],
    "subcluster_infiltrating": ["leiden_sub", "INHBA", "MERTK", "CEACAM8", "FUT4", "ICAM1", "celltype_predictions"],
    "subcluster_cDC2": [
        "leiden_sub", "INHBA", "MERTK", "SPP1", "STAB1", "CD36", "F13A1", "LGMN", "FN1", "CD14", "FCGR3A",
        "HLA-DRB1", "ITGAX", "ITGAM", "CD163", "FCGR1A", "CD36",
        "ENTPD1", "AFF3", "PAK1", "CD1E", "CD1C", "CLEC4A", "FCER1A", "CD207", "CLEC10A", "SIRPA",
        "CLEC9A", "FLT3", "CADM1", "IRF8", "XCR1", "CCR7", "FSCN1", "BIRC3", "CSF3R", "CXCR2",
        "IGF1R", "IGF1", "MKI67", "celltype_predictions",
    ],
    "subcluster_leiden_2": [
        "leiden_sub", "INHBA", "MERTK", "SPP1", "STAB1", "CD36", "F13A1", "LGMN", "FN1", "CD14", "FCGR3A", "VCAN",
        "HLA-DRB1", "ITGAX", "ITGAM", "CD163", "FCGR1A", "CD36",
        "ENTPD1", "AFF3", "PAK1", "CD1E", "CD1C", "CLEC4A", "FCER1A", "CD207", "CLEC10A", "SIRPA",
        "CLEC9A", "FLT3", "CADM1", "IRF8", "XCR1", "CCR7", "FSCN1", "BIRC3", "CSF3R", "CXCR2",
        "IGF1R", "IGF1", "MKI67", "celltype_predictions",
    ],
    "post_refinement": ["CD207", "STAB1", "ITGAX", "CD36", "APOE", "INHBA", "MKI67", "cell_state"],

    "final_panel_1": ["STAB1", "ITGAX", "CD36", "APOE", "MKI67", "CD207", "CLEC9A", "BIRC3",
                      "CSF3R", "CD14", "FCGR3A", "CD163", "ITGAM", "ITGAX", "cell_state"],
    "final_panel_2": ["STAB1", "ITGAX", "CD36", "APOE", "VCAN", "MKI67", "CD207", "CLEC9A", "BIRC3",
                      "CSF3R", "CD14", "FCGR3A", "CD163", "IGF1R", "ITGAM", "ITGAX", "IL4R", "IL13RA1",
                      "CD33", "ACP5", "MMP9", "cell_state"],
                      
    "coarse": ["cell_state_v2", "MERTK", "INHBA"],
    "publication": [             # saved as umap_genes.pdf
        "INHBA", "MERTK", "MARCO", "ITGAX", "FABP4", "PPARG", "CD36", "MAFB", "SPP1", "CD1C", "BIRC3",
        "MRC1", "SIGLEC1", "CX3CR1", "SIGLEC1", "ITGAM", "APOE", "TREM2", "STAB1", "CLEC9A", "MKI67",
        "HLA-DRA", "HLA-DRB1", "HLA-DRB1", "HLA-DQA1", "FCGR3A", "CCR2", "CD14",
        "CCL17", "TGFB1", "MMP12", "MMP14", "TSPAN4", "CXCL9",
    ],

    "publication_2": ["VCAM1", "FCN1", "SRGAP1", "KIT", "CD52", "CD74", "LYZ", "CD33", "VCAN", "CD14"],
}


label_keys = {"leiden_scvi", "leiden_sub", "cell_state", "cell_state_v2", "celltype_predictions"}  # panel entries that are obs labels
min_detection_pct = 1.0          # genes detected in < this % of cells are flagged by the gene sanity check
reference_annotation_path = None  # e.g. "sub_Myeloid_raw_scvi_combine_annotated_.h5ad" to compare labels with the original run
qc_umap_keys = ["phase", "n_genes", "percent_mito", "recist_v3", "ID", "GEP_score", "celltype_predictions"]

for d in (figures_output_dir, tables_output_dir):
    os.makedirs(d, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, figsize=(4, 4))

## Helpers

In [ ]:
def check_object(ad, stage):
    """One consolidated sanity check, run at each stage instead of ad-hoc prints."""
    print(f"[{stage}] {ad.n_obs} cells x {ad.n_vars} genes | samples: {ad.obs[sample_col].nunique()}")
    for col in ("cell_state", "cell_state_v2"):
        if col in ad.obs:
            n_na = ad.obs[col].isna().sum()
            print(f"  {col}: {ad.obs[col].nunique()} labels, {n_na} unlabelled")
            assert n_na == 0, f"{col} has unlabelled cells"

def assert_raw_counts(X, n=1000):
    sample = X[:n].toarray() if sp.issparse(X) else np.asarray(X[:n])
    assert np.all(sample >= 0) and np.allclose(sample, np.round(sample)), "expected raw integer counts"

def tfidf_ranking(ad, groupby, method="wilcoxon"):
    """Binarise expression, TF-IDF transform, then rank genes per group."""
    binary = (ad.X > 0).astype(np.float32) if sp.issparse(ad.X) else sp.csr_matrix(ad.X > 0, dtype=np.float32)
    ad_bin = anndata.AnnData(X=TfidfTransformer().fit_transform(binary.T).T.tocsr(), obs=ad.obs.copy(), var=ad.var.copy())
    sc.tl.rank_genes_groups(ad_bin, groupby=groupby, use_raw=False, method=method)
    return ad_bin

def rank_markers(ad, groupby, n_genes=20):
    """Logistic-regression and TF-IDF markers per group, unfiltered and specificity-filtered."""
    if isinstance(ad.obs[groupby].dtype, pd.CategoricalDtype):
        ad.obs[groupby] = ad.obs[groupby].cat.remove_unused_categories()
    sc.tl.rank_genes_groups(ad, groupby, method="logreg", key_added="logreg", max_iter=1000)
    ad_tfidf = tfidf_ranking(ad, groupby)
    sc.tl.filter_rank_genes_groups(ad, min_in_group_fraction=0.2, max_out_group_fraction=0.2,
                                   key="logreg", key_added="logreg_filtered")
    sc.tl.filter_rank_genes_groups(ad_tfidf, min_in_group_fraction=0.2, max_out_group_fraction=0.2,
                                   key_added="tfidf_filtered")
    for obj, key in ((ad, "logreg"), (ad, "logreg_filtered"), (ad_tfidf, "rank_genes_groups"), (ad_tfidf, "tfidf_filtered")):
        sc.pl.rank_genes_groups_dotplot(obj, groupby=groupby, n_genes=n_genes, key=key,
                                        standard_scale="var", dendrogram=False, title=key)
    return ad_tfidf

def present(genes, ad):
    if isinstance(genes, dict):
        return {k: [g for g in v if g in ad.var_names] for k, v in genes.items() if any(g in ad.var_names for g in v)}
    return [g for g in genes if g in ad.var_names]

def plot_gene_panel(ad, name, save=None, **kwargs):
    """UMAP of one configured panel; genes/obs keys absent from `ad` are skipped (reported by the sanity check)."""
    keys = [k for k in dict.fromkeys(gene_panels[name]) if k in ad.var_names or k in ad.obs.columns]
    params = dict(size=30, cmap="OrRd", ncols=4, wspace=0.3) | kwargs
    sc.pl.umap(ad, color=keys, save=save, **params)

def gene_panel_sanity(ad):
    """Per panel and gene: present in the data?, duplicated in the list?, % of cells detected."""
    X = ad.layers[counts_layer]
    rows = []
    panels = {**gene_panels, **{f"dotplot:{k}": v for k, v in marker_genes.items()},
              **{f"dotplot_neut:{k}": v for k, v in marker_genes_neutrophil.items()}}
    for panel, keys in panels.items():
        counts = pd.Series(keys).value_counts()
        for gene in dict.fromkeys(keys):
            if gene in ad.obs.columns or gene in label_keys:   # label columns, created later in the notebook
                continue
            in_data = gene in ad.var_names
            pct = float((X[:, ad.var_names.get_loc(gene)] > 0).mean() * 100) if in_data else np.nan
            rows.append({"panel": panel, "gene": gene, "in_data": in_data, "duplicated_in_list": counts[gene] > 1,
                         "pct_cells_detected": pct, "low_detection": in_data and pct < min_detection_pct})
    return pd.DataFrame(rows)

def proportion_barplot(ad, which_var, by, colors=None, order=None, category_order=None,
                       horizontal=False, figsize=(11, 2.5), save_name=None):
    """Stacked % of `which_var` within each level of `by`."""
    data = pd.crosstab(ad.obs[by], ad.obs[which_var], normalize="index") * 100
    if category_order:
        data = data[[c for c in category_order if c in data.columns]]
    if order:
        data = data.reindex([o for o in order if o in data.index])
    color = [colors[c] for c in data.columns] if colors else None
    ax = (data.plot.barh if horizontal else data.plot.bar)(stacked=True, edgecolor="none", figsize=figsize,
                                                           fontsize=12, grid=False, color=color, width=0.85)
    (ax.set_xlim if horizontal else ax.set_ylim)(0, 100)
    ax.set_title(f"{which_var} %") if horizontal else ax.set_ylabel(f"{which_var} %")
    ax.legend(bbox_to_anchor=(1, 1), loc="upper left", frameon=False)
    if save_name:
        ax.figure.savefig(f"{figures_output_dir}/{save_name}", bbox_inches="tight", dpi=300)
    plt.show()
    return data

## 1. Load

In [ ]:
adata = sc.read(input_path)
assert_raw_counts(adata.X)
adata.layers[counts_layer] = adata.X.copy()
for key in (global_rep, subcluster_rep):
    assert key in adata.obsm, f"{key} missing from obsm"
check_object(adata, "loaded")
print(adata.obs[recist_col].value_counts().to_string())

### Sanity check: markers expression

Every gene used in a UMAP panel or marker dotplot: missing from the data, listed more
than once in its panel, or detected in fewer than `min_detection_pct` % of cells
(uninformative on a UMAP). Missing genes are skipped when plotting.

In [ ]:
gene_check = gene_panel_sanity(adata)
gene_check.to_csv(f"{tables_output_dir}/gene_panel_sanity_check.csv", index=False)

summary = gene_check.groupby("panel").agg(
    n_genes=("gene", "size"), n_missing=("in_data", lambda x: (~x).sum()),
    n_duplicated=("duplicated_in_list", "sum"), n_low_detection=("low_detection", "sum"))
display(summary)
print("missing:", sorted(gene_check.loc[~gene_check.in_data, "gene"].unique()))
print("duplicated within a panel:", gene_check.loc[gene_check.duplicated_in_list, ["panel", "gene"]].values.tolist())
print(f"detected in < {min_detection_pct}% of cells:",
      sorted(gene_check.loc[gene_check.low_detection, "gene"].unique()))

## 2. Normalise and cluster (level 1)

Log-normalised once here; all subclusters below are taken from this object and are
not normalised again.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.layers["logcounts"] = adata.X.copy()

sc.pp.neighbors(adata, n_neighbors=global_n_neighbors, use_rep=global_rep, random_state=seed)
sc.tl.umap(adata, random_state=seed)
sc.tl.leiden(adata, key_added="leiden_scvi", resolution=leiden_resolution, random_state=seed, flavor="leidenalg")
sc.pl.umap(adata, color=["leiden_scvi"] + [k for k in qc_umap_keys if k in adata.obs],
           ncols=3, wspace=0.3, size=40, color_map="OrRd")

## 3. Cluster markers

In [ ]:
adata_tfidf = rank_markers(adata, "leiden_scvi")
sc.pl.dotplot(adata, present(marker_genes, adata), groupby="leiden_scvi", use_raw=False,
              standard_scale="var", cmap="Blues", figsize=(43, 7), dendrogram=True)
sc.pl.dotplot(adata, present(marker_genes_neutrophil, adata), groupby="leiden_scvi", use_raw=False,
              standard_scale="var", cmap="Blues", figsize=(12, 6))
plot_gene_panel(adata, "level1_markers", legend_loc="on data", ncols=3)

### Neutrophil contamination check

Fraction of MRC1⁺FABP4⁺ macrophages that also express a neutrophil marker
(CEACAM8 or S100A8); low values argue against neutrophil doublets in the macrophage clusters.

In [ ]:
def expressed(gene):
    return np.asarray(adata[:, gene].X.toarray() if sp.issparse(adata.X) else adata[:, gene].X).ravel() > 0

mac = expressed("MRC1") & expressed("FABP4")
neu = expressed("CEACAM8") | expressed("S100A8")
print(f"neutrophil-marker+ among MRC1+FABP4+ cells: {(mac & neu).sum() / max(mac.sum(), 1):.3f}")

proportion_barplot(adata, "leiden_scvi", sample_col, order=sample_order,
                   save_name=f"barplot_{sample_col}_proportions_leiden_scvi.pdf")

## 4. Level-1 annotation

In [ ]:
# --- level-1 annotation: leiden_scvi cluster -> cell_state ---
level1_labels = {
    "0": "Macrophage CD36high", "1": "Macrophage CD36high", "2": "Macrophage INHBAhigh",
    "3": "Macrophage INHBAhigh", "4": "Macrophage INHBAhigh", "5": "Macrophage INHBAhigh",
    "6": "cDC2", "7": "Macrophage STAB1high", "8": "Macrophage CD36high",
    "9": "Macrophage INHBAhigh_infiltrating", "10": "Myeloid_dividing", "11": "Mast cell",
    "12": "cDC1_activatedDC", "13": "Macrophage INHBAhigh_inflammatory", "14": "Monocyte_CD16high",
    "15": "MDSC",
}

unmapped = set(adata.obs["leiden_scvi"].astype(str)) - set(level1_labels)
assert not unmapped, f"leiden clusters without a label: {sorted(unmapped, key=int)} (cluster IDs changed?)"
adata.obs["cell_state"] = adata.obs["leiden_scvi"].astype(str).map(level1_labels)
sc.pl.umap(adata, color=["cell_state", "leiden_scvi"], size=30)
plot_gene_panel(adata, "level1_check", size=35)
check_object(adata, "level 1")

## 5. Subclustering refinements

Each refinement reembeds the selected cells on `subcluster_rep`, reclusters, shows
markers, relabels subclusters and writes the labels back to the full object.

In [ ]:
def refine(ad, spec):
    mask = ad.obs[spec["select_col"]].astype(str).isin(spec["select_values"])
    if mask.sum() <= subcluster_n_neighbors:
        raise ValueError(f"{spec['name']}: only {mask.sum()} cells selected (cluster IDs changed?)")
    sub = ad[mask].copy()
    sc.pp.neighbors(sub, n_neighbors=subcluster_n_neighbors, use_rep=subcluster_rep, random_state=seed)
    sc.tl.umap(sub, random_state=seed)
    sc.tl.leiden(sub, key_added="leiden_sub", resolution=leiden_resolution, random_state=seed, flavor="leidenalg")
    print(f"=== {spec['name']}: {sub.n_obs} cells, {sub.obs['leiden_sub'].nunique()} subclusters")

    sc.pl.umap(sub, color=["leiden_sub"] + [k for k in ("phase", "scrublet_doublet", "n_genes", "percent_mito")
                                             if k in sub.obs], ncols=3, wspace=0.3, size=15, color_map="OrRd")
    rank_markers(sub, "leiden_sub", n_genes=10)
    sc.pl.dotplot(sub, present(marker_genes, sub), groupby="leiden_sub", use_raw=False,
                  standard_scale="var", cmap="Blues", figsize=(40, 7))
    if spec.get("umap_panel"):
        plot_gene_panel(sub, spec["umap_panel"], ncols=3, size=25)
    if spec["labels"] is None:                     # exploratory: inspect only, no relabelling
        print("exploratory only - labels unchanged\n")
        return

    unmapped = set(sub.obs["leiden_sub"].astype(str)) - set(spec["labels"])
    assert not unmapped, f"{spec['name']}: subclusters without a label: {sorted(unmapped, key=int)}"
    new_labels = sub.obs["leiden_sub"].astype(str).map(spec["labels"])
    ad.obs.loc[new_labels.index, "cell_state"] = new_labels.values
    sub.obs["cell_state_refined"] = new_labels.astype("category")
    sc.pl.umap(sub, color="cell_state_refined", size=25)
    print(new_labels.value_counts().to_string(), "\n")
    plot_gene_panel(ad, "post_refinement", size=25, cmap="RdPu")

# --- subclustering refinements, applied in order; each selects cells and relabels subclusters ---
# not shown
# refinements []

adata.obs["cell_state"] = adata.obs["leiden_scvi"].astype(str).map(level1_labels)   # reset: safe to rerun
for spec in refinements:
    refine(adata, spec)


cell_state_order = [
    "Macrophage INHBAhigh", "Macrophage CD36high", "Macrophage STAB1high", "cDC2",
    "Macrophage INHBAhigh_infiltrating", "Myeloid_dividing", "Mast cell", "cDC1_activatedDC",
    "Macrophage INHBAhigh_inflammatory", "Monocyte_CD16high", "MDSC",
]


adata.obs["cell_state"] = pd.Categorical(adata.obs["cell_state"], categories=cell_state_order)
check_object(adata, "refined")
sc.pl.umap(adata, color="cell_state", palette=cell_state_colors, size=30)

In [ ]:
plot_gene_panel(adata, "final_panel_1", cmap="RdPu")
plot_gene_panel(adata, "final_panel_2", cmap="RdPu")

## 6. Coarse labels and publication UMAPs

In [ ]:
# --- coarse labels: cell_state -> cell_state_v2 (unlisted states keep their name) ---
coarse_labels = {
    "Macrophage CD36high": "Macrophage MERTKhigh",
    "Macrophage STAB1high": "Macrophage MERTKhigh",
    "Macrophage INHBAhigh_inflammatory": "Macrophage INHBAhigh",
    "Macrophage INHBAhigh_infiltrating": "Macrophage INHBAhigh",
}


cell_state_colors = {
    "Macrophage INHBAhigh": "#d33f6a", "Macrophage STAB1high": "#99103d", "Macrophage CD36high": "#9c9ede",
    "Macrophage INHBAhigh_infiltrating": "#e7969c", "Macrophage INHBAhigh_inflammatory": "#54102a",
    "cDC2": "grey", "cDC1_activatedDC": "#bcbbbb", "Monocyte_CD16high": "#96e7e1", "MDSC": "#af7287",
    "Myeloid_dividing": "#D1E2C4", "Mast cell": "#f44040",
}

cell_state_v2_order = [
    "Macrophage INHBAhigh", "Macrophage MERTKhigh", "cDC2", "Myeloid_dividing", "Mast cell",
    "cDC1_activatedDC", "Monocyte_CD16high", "MDSC",
]

cell_state_v2_colors = {
    "Macrophage INHBAhigh": "#d33f6a", "Macrophage MERTKhigh": "#54102a", "cDC2": "grey",
    "cDC1_activatedDC": "#bcbbbb", "Monocyte_CD16high": "#96e7e1", "MDSC": "#af7287",
    "Myeloid_dividing": "#D1E2C4", "Mast cell": "#f44040",
}

#----Plot Figure 3----#

adata.obs["cell_state_v2"] = pd.Categorical(
    adata.obs["cell_state"].astype(str).replace(coarse_labels), categories=cell_state_v2_order)
check_object(adata, "coarse labels")
plot_gene_panel(adata, "coarse", size=20, palette=cell_state_v2_colors)

sc.set_figure_params(figsize=(8, 8))
is_liver = adata.obs["ID"].isin(liver_biopsy_ids)
for label, colors in (("cell_state", cell_state_colors), ("cell_state_v2", cell_state_v2_colors)):
    for subset, suffix in ((adata, ""), (adata[~is_liver], "_excl_liverBiopsy"), (adata[is_liver], "_only_liverBiopsy")):
        sc.pl.umap(subset, color=label, size=80 if suffix == "" else 100, palette=colors,
                   save=f"_{label}{suffix}_final.pdf")

sc.set_figure_params(figsize=(4, 4))
plot_gene_panel(adata, "publication", save="_genes.pdf", frameon=False, cmap="RdPu")
sc.set_figure_params(figsize=(3, 3))
plot_gene_panel(adata, "publication_2", frameon=False, cmap="RdPu", vmax=5)
sc.set_figure_params(figsize=(4, 4))

### Final-label markers

In [ ]:
for label, order in (("cell_state", cell_state_order), ("cell_state_v2", cell_state_v2_order)):
    rank_markers(adata, label)
    sc.pl.dotplot(adata, present(marker_genes, adata), groupby=label, use_raw=False,
                  standard_scale="var", cmap="Blues", figsize=(30, 7))

## 7. Composition per sample and treatment stage

In [ ]:
# myeloid cells per sample, with a y-axis break #----Plot Figure 3----#
cell_counts = adata.obs[sample_col].value_counts().reindex([s for s in sample_order if s in set(adata.obs[sample_col])])
fig, (ax_top, ax_bot) = plt.subplots(2, 1, sharex=True, figsize=(10, 5.1), gridspec_kw={"height_ratios": [1, 2], "hspace": 0.05})
for ax in (ax_top, ax_bot):
    ax.bar(range(len(cell_counts)), cell_counts.values, color="black", width=0.8)
    ax.grid(False)
ax_top.set_ylim(count_axis_break, cell_counts.max() * 1.1)
ax_bot.set_ylim(0, count_axis_break)
ax_top.spines["bottom"].set_visible(False); ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(bottom=False)
d = 0.015
for ax, y in ((ax_top, 0), (ax_bot, 1)):
    for x in (0, 1):
        ax.plot((x - d, x + d), (y - d, y + d), transform=ax.transAxes, color="k", clip_on=False)
ax_bot.set_xticks(range(len(cell_counts)), cell_counts.index, rotation=90)
ax_bot.set_ylabel("Myeloid cells")
fig.savefig(f"{figures_output_dir}/barplot_{sample_col}_myeloid_cell_counts_ybreak.pdf", bbox_inches="tight", dpi=300)
plt.show()

proportion_barplot(adata, "cell_state", sample_col, colors=cell_state_colors, order=sample_order,
                   category_order=cell_state_order, save_name=f"barplot_{sample_col}_proportions_cell_state.pdf")
proportion_barplot(adata, "cell_state", recist_col, colors=cell_state_colors, category_order=cell_state_order,
                   horizontal=True, figsize=(6, 2), save_name=f"barplot_{recist_col}_proportions_cell_state.pdf")
proportion_barplot(adata, "cell_state_v2", recist_col, colors=cell_state_v2_colors, category_order=cell_state_v2_order,
                   horizontal=True, figsize=(4, 2), save_name=f"barplot_{recist_col}_proportions_cell_state_v2.pdf")

### Reference-label agreement (cells with prediction probability ≥ 0.95)

In [ ]:
if {"probabilities", "celltype_predictions"} <= set(adata.obs.columns):
    confident = adata.obs["probabilities"] >= 0.95
    print(f"{confident.sum()} / {adata.n_obs} cells with probability >= 0.95")
    sc.pl.umap(adata, color=["probabilities", "celltype_predictions"])
    sc.pl.umap(adata[confident], color=["probabilities", "celltype_predictions"], size=35)
    agreement = pd.crosstab(adata.obs.loc[confident, "cell_state"], adata.obs.loc[confident, "celltype_predictions"])
    agreement.to_csv(f"{tables_output_dir}/cell_state_vs_reference_predictions.csv")
    display(agreement)

## 8. Save

In [ ]:
for key in [k for k in adata.uns if k.endswith("_filtered")]:
    del adata.uns[key]                        # filtered rankings contain NaN names and fail to write

adata.obs[["leiden_scvi", "cell_state", "cell_state_v2"]].to_csv(annotation_csv)
adata.X = adata.layers[counts_layer].copy()
adata.uns.pop("log1p", None)
adata.write(output_path, compression="gzip")
check_object(adata, "saved")